# Train the matcher

Trains the XGBoost matcher on candidate-pair features (Apache-2.0 licensed, far under the 8B-parameter cap) and picks a decision threshold that maximizes macro F_0.5 on a held-out validation split of Source 1 entities.

**Prerequisite:** run from a terminal first —
```bash
python -m src.build_candidates --split train
python -m src.build_features --split train
```
This notebook only trains/evaluates/saves the model; it does not regenerate candidates or features.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import config
from src.io_utils import read_source, read_ground_truth
from src.features import FEATURE_COLUMNS
from src.train_matcher import (
    split_entities_by_source1,
    train_xgboost,
    score_candidates,
    save_model,
)
from src.evaluation import find_best_threshold, macro_f0_5, predictions_from_scores

## Load features + ground truth

In [ ]:
features_path = config.FEATURES_DIR / 'train_features.parquet'
assert features_path.exists(), f'{features_path} missing — run build_features.py --split train first'

features = pd.read_parquet(features_path)
s1 = read_source(config.TRAIN_SOURCE1)
gt = read_ground_truth(config.TRAIN_GROUND_TRUTH)
truth = {r.source1_entity_id: set(r.matched_entity_ids) for r in gt.itertuples(index=False)}

print(f'{len(features):,} labeled candidate pairs, {features["label"].sum():,} positive')

## Entity-level train/validation split

Split by Source 1 entity (not by row) so every candidate pair for one entity stays on the same side — otherwise validation would leak.

In [ ]:
train_ids, val_ids = split_entities_by_source1(s1['entity_id'])
train_df = features[features['source1_entity_id'].isin(train_ids)]
val_df = features[features['source1_entity_id'].isin(val_ids)]
print(f'train pairs={len(train_df):,} ({len(train_ids):,} entities) | val pairs={len(val_df):,} ({len(val_ids):,} entities)')

## Train

Tune `n_estimators` / `max_depth` / `learning_rate` here if needed.

In [ ]:
model = train_xgboost(train_df)
pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)

## Pick the decision threshold

F_0.5 weights precision 2x over recall — the grid search below finds the probability cutoff that maximizes macro F_0.5 on the validation split, which is usually well above the naive 0.5.

In [ ]:
scored_val = score_candidates(model, val_df)
val_truth = {s1_id: truth.get(s1_id, set()) for s1_id in val_ids}

best_threshold, best_f05 = find_best_threshold(scored_val, val_truth)
print(f'best threshold={best_threshold:.3f}  validation macro F0.5={best_f05:.4f}')

## Error analysis

Look at a sample of false positives and false negatives at the chosen threshold.

In [ ]:
preds = predictions_from_scores(scored_val, best_threshold)

fp, fn = [], []
for s1_id, true_ids in val_truth.items():
    pred_ids = preds.get(s1_id, set())
    fp += [(s1_id, cid) for cid in pred_ids - true_ids]
    fn += [(s1_id, cid) for cid in true_ids - pred_ids]

print(f'false positives: {len(fp)} | false negatives: {len(fn)}')
scored_val.set_index(['source1_entity_id', 'candidate_entity_id']).loc[fp[:10]] if fp else 'none'

## Save the model

Written to `../models/` (gitignored) — `src/predict.py` loads it from there.

In [ ]:
save_model(model, best_threshold)
print(f'saved model to {config.MATCHER_MODEL_PATH}')
print(f'saved threshold {best_threshold:.3f} to {config.THRESHOLD_PATH}')

## Next steps

```bash
python -m src.build_candidates --split test
python -m src.build_features --split test
python -m src.predict
```
Then validate with the command in `../../CLAUDE.md`.